# triangle-barycentric composite — cx21: build the 3x3 barycentric matrix via t.stack along the column axis

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `triangle-barycentric`, `stack-vs-cat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "triangle-barycentric"
DD_ATOM_IDS = ["triangle-barycentric", "stack-vs-cat"]
DD_SUBTOPICS = ["Geometry: Barycentric coords", "PyTorch: stack vs cat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The barycentric 3x3 system has the shape `[-D | (B-A) | (C-A)] [s; u; v] = O - A`. Each column is one `(3,)` vector, and the matrix is built by GLUING those three vectors along a NEW axis (the column axis).

This is exactly the `torch.stack` vs `torch.cat` distinction:
  - `t.stack([v1, v2, v3], dim=1)` → `(3, 3)` with each `v_i` as a COLUMN (new axis at dim 1).
  - `t.cat([v1, v2, v3], dim=0)` → `(9,)` — concatenated along an EXISTING axis (wrong shape).

Picking the wrong one is the most common bug in the ARENA implementation of this. The drill forces the correct choice.

### Composite Exercise — build the 3x3 barycentric matrix via t.stack along the column axis

**Atoms exercised together**: `triangle-barycentric`, `stack-vs-cat`

Implement `cx21_barycentric_matrix(D, A, B, C)` that returns the 3x3 matrix whose columns are `[-D, B - A, C - A]`.

- All inputs are `(3,)` tensors.
- Output shape: `(3, 3)`. Column 0 is `-D`, column 1 is `B - A`, column 2 is `C - A`.

Use `torch.stack` (NOT `torch.cat`) along `dim=1` so the three vectors become columns. The test asserts shape `(3, 3)` and that each column equals the expected vector — and also that `M @ t.tensor([s, u, v]) == O - A` for a known solution (i.e. the matrix is correctly assembled for the barycentric system).

In [ ]:
def cx21_barycentric_matrix(D, A, B, C):
    # Atom A (triangle-barycentric): columns are [-D, B - A, C - A].
    col0 = -D
    col1 = B - A
    col2 = C - A
    # Atom B (stack-vs-cat): STACK along a NEW dim=1 to make each (3,) a column.
    # Using cat would concat along an existing axis -> (9,), wrong shape.
    return t.stack([col0, col1, col2], dim=1)


<details><summary>Show solution — cx21</summary>

```python
def cx21_barycentric_matrix(D, A, B, C):
    # Atom A (triangle-barycentric): columns are [-D, B - A, C - A].
    col0 = -D
    col1 = B - A
    col2 = C - A
    # Atom B (stack-vs-cat): STACK along a NEW dim=1 to make each (3,) a column.
    # Using cat would concat along an existing axis -> (9,), wrong shape.
    return t.stack([col0, col1, col2], dim=1)
```

`t.stack(..., dim=1)` creates a new axis at position 1, turning three `(3,)` vectors into a `(3, 3)` matrix with each vector as a column. `t.cat(..., dim=0)` would concatenate them along the existing axis 0 and give `(9,)` — a flat vector, not a matrix. The barycentric system needs the new-axis variant; this is the canonical stack-vs-cat call.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx21'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx21',
        'subtopics': ["Geometry: Barycentric coords", "PyTorch: stack vs cat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()